In [18]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

df = pd.read_csv('train (2).csv')


In [3]:
df.columns  #id not needed

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [14]:
df.shape

(1460, 81)

In [6]:
for x in df.columns:
 count = df[x].isnull().sum()
 if count>0:
  print(f"{x} with null {count}")

LotFrontage with null 259
Alley with null 1369
MasVnrType with null 872
MasVnrArea with null 8
BsmtQual with null 37
BsmtCond with null 37
BsmtExposure with null 38
BsmtFinType1 with null 37
BsmtFinType2 with null 38
Electrical with null 1
FireplaceQu with null 690
GarageType with null 81
GarageYrBlt with null 81
GarageFinish with null 81
GarageQual with null 81
GarageCond with null 81
PoolQC with null 1453
Fence with null 1179
MiscFeature with null 1406


In [32]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=67)
ordinal_categories = {
    'Alley':['Grvl','Pave'],
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'Utilities': ['AllPub', 'NoSewr', 'NoSeWa', 'ELO'],
    'LandSlope': ['Gtl', 'Mod', 'Sev'],
    'OverallQual': ['Very Poor', 'Poor', 'Fair', 'Below Average', 'Average',
                    'Above Average', 'Good', 'Very Good', 'Excellent', 'Very Excellent'],
    'ExterQual': ['Po', 'Fa', 'Ta', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'Ta', 'Gd', 'Ex'],
    'BsmtQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['Fa', 'TA', 'Gd', 'Ex'],
}

# 2. AUTO-DETECT COLUMN TYPES
def detect_column_types(df, ordinal_cols):
    all_cols = df.columns.tolist()
    cat_cols = [col for col in all_cols if col not in ordinal_cols and df[col].dtype == 'object']
    num_cols = [col for col in all_cols if col not in ordinal_cols and col not in cat_cols]
    return cat_cols, num_cols

# 3. BUILD PREPROCESSOR
def build_preprocessor(df, ordinal_categories):
    ordinal_cols = list(ordinal_categories.keys())
    cat_cols, num_cols = detect_column_types(df, ordinal_cols)

    ordinal_encoder = OrdinalEncoder(
        categories=[ordinal_categories[col] for col in ordinal_cols],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )

    return ColumnTransformer([
        ('ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', ordinal_encoder)
        ]), ordinal_cols),

        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols),

        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols)
    ], remainder='passthrough')

def build_full_pipeline(df, ordinal_categories):
    preprocessor = build_preprocessor(df, ordinal_categories)
    model = XGBRegressor(random_state=67,n_jobs=-1, verbosity =0)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
# param_dist = {
#     'regressor__n_estimators': [100, 200, 300, 500],
#     'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'regressor__max_depth': [3, 5, 7, 9],
#     'regressor__subsample': [0.6, 0.8, 1.0],
#     'regressor__colsample_bytree': [0.6, 0.8, 1.0],
#     'regressor__reg_alpha': [0, 0.1, 1, 10],
#     'regressor__reg_lambda': [1, 10, 100],
# }

# pipeline = build_full_pipeline(X_train, ordinal_categories)
# randomsearch = RandomizedSearchCV(
#     pipeline,
#     param_dist,
#     n_iter = 30,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     random_state=67,
#     n_jobs=-1,
#     verbose = 1
# )
# randomsearch.fit(X_train, Y_train)
# for param,value in randomsearch.best_params_.items():
#  print(f"{param} : {value}")
# y_pred = randomsearch.best_estimator_.predict(X_test)
# r2 = r2_score(Y_test, y_pred)
# mse = mean_squared_error(Y_test, y_pred)
# rmse = np.sqrt(mse)
# print(f"R2 Score: {r2}")
# print(f"MSE: {mse}")
# print(f"RMSE: {rmse}")

In [34]:
final_model = XGBRegressor(
   subsample = 0.8,
reg_lambda = 1,
reg_alpha = 0.1,
n_estimators= 300,
max_depth= 5,
learning_rate= 0.05,
colsample_bytree= 0.6,
)

def final_pipeline(df,ordinal_categories):
    preprocessor = build_preprocessor(df, ordinal_categories)
    model = final_model
    return Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
final_models = final_pipeline(X_train, ordinal_categories)
final_models.fit(X_train, Y_train)
y_pred_final = final_models.predict(X_test)
r2_final = r2_score(Y_test, y_pred_final)
mse_final = mean_squared_error(Y_test, y_pred_final)
rmse_final = np.sqrt(mse_final)


print(f"R² Score: {r2_final:.4f}")
print(f"MSE: {mse_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")


R² Score: 0.9171
MSE: 433871264.0000
RMSE: 20829.5767


In [22]:
df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [14]:
df['LotFrontage'].value_counts()

,count
LotFrontage,
60.0,143
70.0,70
80.0,69
50.0,57
75.0,53
...,...
182.0,1
160.0,1
152.0,1


In [41]:
df_test = pd.read_csv('test (2).csv')

test_ids = df_test['Id']

X_test_final = df_test.drop(columns=['Id'])

for col in X_train.columns:
    if col not in X_test_final.columns:
        X_test_final[col] = np.nan

X_test_final = X_test_final[X_train.columns]

predictions = final_models.predict(X_test_final)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

# 8. Save
submission.to_csv('submission.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f" Predictions range: {predictions.min():.2f} - {predictions.max():.2f}")

assert len(predictions) == len(test_ids), "Length mismatch!"
print(" All checks passed! Ready to submit!")

Submission shape: (1459, 2)
 Predictions range: 35772.37 - 593566.75
 All checks passed! Ready to submit!
